# Using of Pipelines and Column Transformations from start to end till making model files

### 1. Import Everything

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error

import joblib

### 2. Load Dataset

In [8]:
def load_data():
    df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
    return df

### 3. Preprocessing Builder Functions

In [11]:
# we perform this inorder to increase reusability and control on program
def build_preprocessor(numerical_cols, categorical_cols):
    
    # Numerical Pipeline
    num_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    # Categorical Pipeline
    cat_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    # Combine
    preprocessor = ColumnTransformer(transformers=[
        ("num", num_pipeline, numerical_cols),
        ("cat", cat_pipeline, categorical_cols)
    ])
    
    return preprocessor

### 4. Model  Pipeline Builder

In [14]:
def build_pipeline(model, numerical_cols, categorical_cols):
    
    preprocessor = build_preprocessor(numerical_cols, categorical_cols)
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    return pipeline

## 5. Building Models on different type of Problems - Training + Tuning

### Logistic Regression (Classification) 

In [18]:
def train_logistic_regression(X, y, numerical_cols, categorical_cols):
    
    model = LogisticRegression(max_iter=1000)
    
    pipeline = build_pipeline(model, numerical_cols, categorical_cols)
    
    param_grid = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    }
    
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
    grid.fit(X, y)
    
    return grid

### Linear Regression

In [25]:
def train_linear_regression(X, y, numerical_cols, categorical_cols):
    
    model = LinearRegression()
    
    pipeline = build_pipeline(model, numerical_cols, categorical_cols)
    
    param_grid = {
        "model__fit_intercept": [True, False]
    }
    
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_squared_error")
    grid.fit(X, y)
    
    return grid

## 6. Main Execution

In [28]:
df = load_data()

features = ["Age", "Fare", "Sex", "Embarked", "Pclass"]
target = "Survived"

X = df[features]
y = df[target]

numerical_cols = ["Age", "Fare"]
categorical_cols = ["Sex", "Embarked", "Pclass"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

grid_model = train_logistic_regression(X_train, y_train, numerical_cols, categorical_cols)

# Predictions
y_pred = grid_model.predict(X_test)

print("Best Parameters:", grid_model.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Accuracy: 0.7877094972067039


### 7. Save Model For Production

In [31]:
joblib.dump(grid_model.best_estimator_, "logistic_model.pkl")

['logistic_model.pkl']